In [1]:
# Start with tinyshakespeare dataset to follow Karpathy for initial development
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-05-09 13:47:25--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M   699KB/s    in 1.6s    

2026-05-09 13:47:27 (699 KB/s) - ‘input.txt’ saved [1115394/1115394]



In [6]:
s = set("aefasdjfkafjslkdflajdf")
len(s)

for i,c in enumerate(s):
    print(f"{i}:{c}")

0:d
1:j
2:e
3:a
4:l
5:k
6:s
7:f


In [19]:
# Simple char based tokenization, attempting compatibility with tiktoken methods for later drop in replacement

class Char_Tokenizer():
    def __init__(self, intext: str):
        self.charset = set(intext)
        sorted_charset = sorted(list(self.charset))
        self.stoi = {c:i for i,c in enumerate(sorted_charset)}
        self.itos = {i:c for i,c in enumerate(sorted_charset)}
    
    def n_vocab(self):
        return len(self.charset)
    
    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]
    
    def decode(self, l: list[int]) -> str:
        return ''.join(self.itos[i] for i in l)


with open("input.txt", 'r', encoding="utf-8") as f:
    text = f.read()
tokenizer = Char_Tokenizer(text)
print(tokenizer.n_vocab())
print(repr(''.join(sorted(list(tokenizer.charset)))))
s = "Hello"
assert(tokenizer.decode(tokenizer.encode(s))==s)


65
"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"


In [22]:
import torch 

data = torch.tensor(tokenizer.encode(text), dtype=torch.long)

# split into train and validation data
split_index = int(0.9*len(data))
train = data[:split_index]
val = data[split_index:]

In [50]:
device = 'mps' if torch.mps.is_available() else 'cpu'

torch.manual_seed(1337)
batch_size = 4
block_size = 8

def batch(data):
    inds = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in inds])
    y = torch.stack([data[i+1:i+block_size+1] for i in inds])
    x,y = x.to(device), y.to(device)
    return x, y

x_train, y_train = batch(train)
print(x_train.shape)
print(x_train)
print(y_train.shape)
print(y_train)

torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='mps:0')
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='mps:0')


In [ ]:
from math import sqrt
from einops import rearrange
from torch import einsum

class MultiHeadAttention(nn.Module):
    
    def __init__(self,n_heads, d_embedding, block_size):
        super().__init__()
        self.n_heads = n_heads

        self.qry_weights = nn.Parameter(torch.randn(d_embedding, d_embedding) / sqrt(d_embedding))
        self.key_weights = nn.Parameter(torch.randn(d_embedding, d_embedding) / sqrt(d_embedding))
        self.val_weights = nn.Parameter(torch.randn(d_embedding, d_embedding) / sqrt(d_embedding))
        self.out_weights = nn.Parameter(torch.randn(d_embedding, d_embedding) / sqrt(d_embedding))

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    
    def forward(self, x):
        #x.shape is (b,l,e) - b is batch size, l is sequence length, e is embedding vector length
        b,l,e = x.shape

        q = einsum('b l e, e d -> b l d', x, self.qry_weights)
        k = einsum('b l e, e d -> b l d', x, self.key_weights)
        v = einsum('b l e, e d -> b l d', x, self.val_weights)

        q = rearrange(q, 'b l (n d) -> b n l d', n=self.n_heads)
        k = rearrange(k, 'b l (n d) -> b n l d', n=self.n_heads)
        v = rearrange(v, 'b l (n d) -> b n l d', n=self.n_heads)

        attn = einsum('b n q d, b n k d -> b n q k', q, k)

        attn = attn / sqrt(q.shape[-1])

        attn = attn.masked_fill(self.tril[:l, :l] == 0, float('-inf'))
        attn = attn.softmax(dim=-1)

        out = einsum('b n q l, b n l d -> b n q d', attn, v)

        out = rearrange(out, 'b n q d -> b q (n d)')
        out = einsum('b l e, e o -> b l o', out, self.out_weights)
        return out

In [ ]:
torch.manual_seed(1337)

b = batch_size #batch size
l = block_size # Sequence length, in training can always use block size
e = 32 # Length of the embedding vector that the vocab gets mapped to